In [ ]:
from google.colab import drive
import sys
import os
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/Food_Hazard_Project/NLP'
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from data.data_processor import DataProcessor
from data.dataset import FoodDataset
from features.tfidf import TFIDF
from models.svm_classifier import SVMClassifier

env_path = os.path.join(PROJECT_PATH, '.env')
load_dotenv(env_path)
hf_token = os.getenv('HF_TOKEN')

In [ ]:
processor = DataProcessor()

data = processor.process_data(
    train_path='csv/train.csv',
    val_path='csv/valid.csv',
    test_path='csv/test.csv',
    tokenize=False
)

train_df = data['train_df']
val_df = data['val_df']
test_df = data['test_df']

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

hazard_weights = compute_class_weight('balanced', classes=np.unique(train_df['hazard_label']), y=train_df['hazard_label'].values)
product_weights = compute_class_weight('balanced', classes=np.unique(train_df['product_label']), y=train_df['product_label'].values)

tensor_hazard_weights = torch.tensor(hazard_weights, dtype=torch.float32).to(device)
tensor_product_weights = torch.tensor(product_weights, dtype=torch.float32).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, predictions, average='macro')}

class DualModelTrainer(Trainer):
    def __init__(self, target_column, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.target_column = target_column
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop(self.target_column)
        other_column = 'product_label' if self.target_column == 'hazard_label' else 'hazard_label'
        if other_column in inputs:
            inputs.pop(other_column)

        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(outputs.logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")

train_dataset_bert = FoodDataset(train_df['clean_text'], train_df['hazard_label'], train_df['product_label'], tokenizer_bert)
val_dataset_bert = FoodDataset(val_df['clean_text'], val_df['hazard_label'], val_df['product_label'], tokenizer_bert)
dummy_labels = pd.Series([0] * len(test_df))
test_dataset_bert = FoodDataset(test_df['clean_text'], dummy_labels, dummy_labels, tokenizer_bert)

model_bert_haz = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=10).to(device)
args_haz = TrainingArguments(output_dir="./res_bert_haz", eval_strategy="epoch", save_strategy="epoch",
                             learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
                             num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1", remove_unused_columns=False, label_names=['hazard_label'])
trainer_bert_haz = DualModelTrainer(target_column='hazard_label', class_weights=tensor_hazard_weights, model=model_bert_haz, args=args_haz, train_dataset=train_dataset_bert, eval_dataset=val_dataset_bert, compute_metrics=compute_metrics)
trainer_bert_haz.train()
bert_hazard_probs = F.softmax(torch.tensor(trainer_bert_haz.predict(test_dataset_bert).predictions), dim=-1).numpy()

model_bert_prod = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=22).to(device)
args_prod = TrainingArguments(output_dir="./res_bert_prod", eval_strategy="epoch", save_strategy="epoch",
                              learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
                              num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1", remove_unused_columns=False, label_names=['product_label'])
trainer_bert_prod = DualModelTrainer(target_column='product_label', class_weights=tensor_product_weights, model=model_bert_prod, args=args_prod, train_dataset=train_dataset_bert, eval_dataset=val_dataset_bert, compute_metrics=compute_metrics)
trainer_bert_prod.train()
bert_product_probs = F.softmax(torch.tensor(trainer_bert_prod.predict(test_dataset_bert).predictions), dim=-1).numpy()

del model_bert_haz, model_bert_prod, trainer_bert_haz, trainer_bert_prod, train_dataset_bert, val_dataset_bert
torch.cuda.empty_cache()
gc.collect()

In [ ]:
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")

train_dataset_rob = FoodDataset(train_df['clean_text'], train_df['hazard_label'], train_df['product_label'], tokenizer_roberta)
val_dataset_rob = FoodDataset(val_df['clean_text'], val_df['hazard_label'], val_df['product_label'], tokenizer_roberta)
test_dataset_rob = FoodDataset(test_df['clean_text'], dummy_labels, dummy_labels, tokenizer_roberta)

model_rob_haz = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=10).to(device)
args_haz_rob = TrainingArguments(output_dir="./res_rob_haz", eval_strategy="epoch", save_strategy="epoch",
                             learning_rate=1.5e-5, warmup_ratio=0.1, lr_scheduler_type="cosine", per_device_train_batch_size=16, per_device_eval_batch_size=16,
                             num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1", remove_unused_columns=False, label_names=['hazard_label'])
trainer_rob_haz = DualModelTrainer(target_column='hazard_label', class_weights=tensor_hazard_weights, model=model_rob_haz, args=args_haz_rob, train_dataset=train_dataset_rob, eval_dataset=val_dataset_rob, compute_metrics=compute_metrics)
trainer_rob_haz.train()
roberta_hazard_probs = F.softmax(torch.tensor(trainer_rob_haz.predict(test_dataset_rob).predictions), dim=-1).numpy()

model_rob_prod = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=22).to(device)
args_prod_rob = TrainingArguments(output_dir="./res_rob_prod", eval_strategy="epoch", save_strategy="epoch",
                              learning_rate=1.5e-5, warmup_ratio=0.1, lr_scheduler_type="cosine", per_device_train_batch_size=16, per_device_eval_batch_size=16,
                              num_train_epochs=15, load_best_model_at_end=True, metric_for_best_model="macro_f1", remove_unused_columns=False, label_names=['product_label'])
trainer_rob_prod = DualModelTrainer(target_column='product_label', class_weights=tensor_product_weights, model=model_rob_prod, args=args_prod_rob, train_dataset=train_dataset_rob, eval_dataset=val_dataset_rob, compute_metrics=compute_metrics)
trainer_rob_prod.train()
roberta_product_probs = F.softmax(torch.tensor(trainer_rob_prod.predict(test_dataset_rob).predictions), dim=-1).numpy()

del model_rob_haz, model_rob_prod, trainer_rob_haz, trainer_rob_prod, train_dataset_rob, val_dataset_rob
torch.cuda.empty_cache()
gc.collect()    

In [ ]:
tfidf = TFIDF(max_features=15000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

svm_haz = SVMClassifier(C=5.0)
svm_haz.train(X_train_tfidf, train_df['hazard_label'])
svm_hazard_probs = svm_haz.predict_proba(X_test_tfidf)

svm_prod = SVMClassifier(C=5.0)
svm_prod.train(X_train_tfidf, train_df['product_label'])
svm_product_probs = svm_prod.predict_proba(X_test_tfidf)

In [ ]:
W_BERT = 0.35
W_ROB = 0.35
W_SVM = 0.30

ens_haz_probs = (W_BERT * bert_hazard_probs) + (W_ROB * roberta_hazard_probs) + (W_SVM * svm_hazard_probs)
ens_prod_probs = (W_BERT * bert_product_probs) + (W_ROB * roberta_product_probs) + (W_SVM * svm_product_probs)

final_haz_preds = np.argmax(ens_haz_probs, axis=-1)
final_prod_preds = np.argmax(ens_prod_probs, axis=-1)

final_haz_text = data['le_hazard'].inverse_transform(final_haz_preds)
final_prod_text = data['le_product'].inverse_transform(final_prod_preds)

ensemble_holy_trinity = pd.DataFrame({
    'id': test_df.index, 
    'hazard-category': final_haz_text,
    'product-category': final_prod_text
})

sub_path = os.path.join('csv', 'holy_trinity_60_40_submission.csv')
ensemble_holy_trinity.to_csv(sub_path, index=False)